# 🎬 Rick's Clips — Ultra-Fast Google Colab GPU Renderer
Renders full 1080p (1080x1920) Rick's Clips Shorts using Google's free Nvidia T4 Cloud GPU.

---
### ⚡ Quick Start (2 Steps):
1. **Enable Free GPU**: Top menu -> `Runtime` -> `Change runtime type` -> select **T4 GPU** -> click `Save`.
2. **Click Run**: Select your episode (e.g. `emu-war`), set workers to 4, and click the **▶ Play button**.

Rendering runs across 4 parallel GPU workers and downloads the finished 1080p MP4 directly to your PC.

In [ ]:
#@title 🚀 1-Click Ultra-Fast GPU Render: Select Episode & Click Play
import os, sys, time, json, subprocess, glob
from google.colab import files

EPISODE = "emu-war" #@param ["emu-war", "boston-molasses", "lake-peigneur"]
SCALE = 1.0 #@param {type:"slider", min:0.5, max:1.0, step:0.1}
WORKERS = 4 #@param {type:"slider", min:1, max:4, step:1}

print("="*65)
print(f"🎬 FAST GPU RENDER: {EPISODE.upper()} ({WORKERS} parallel workers @ scale {SCALE})")
print("="*65)

# 1. Verify GPU
print("
[1/6] 🔍 Checking Nvidia GPU...")
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

# 2. Configure Nvidia Hardware EGL Driver
print("
[2/6] ⚙️ Linking Tesla T4 Hardware WebGL driver...")
!echo "/usr/lib64-nvidia" > /etc/ld.so.conf.d/nvidia.conf && ldconfig
!mkdir -p /usr/share/glvnd/egl_vendor.d
with open('/usr/share/glvnd/egl_vendor.d/10_nvidia.json', 'w') as f:
    f.write('''{
    "file_format_version" : "1.0.0",
    "ICD" : {
        "library_path" : "/usr/lib64-nvidia/libEGL_nvidia.so.0"
    }
}''')

os.environ['LD_LIBRARY_PATH'] = '/usr/lib64-nvidia:' + os.environ.get('LD_LIBRARY_PATH', '')
os.environ['__GLX_VENDOR_LIBRARY_NAME'] = 'nvidia'
os.environ['__EGL_VENDOR_LIBRARY_FILENAMES'] = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'

# 3. Pull Repository & Install Dependencies
print("
[3/6] 📦 Pulling latest episode code & browser engine...")
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1
!apt-get install -y nodejs ffmpeg xvfb > /dev/null 2>&1

if not os.path.exists("/content/ricks-clips"):
    !git clone https://github.com/pradeep142536/ricks-clips-00.git /content/ricks-clips
else:
    !git -C /content/ricks-clips fetch origin
    !git -C /content/ricks-clips reset --hard origin/main

!cd /content/ricks-clips/templates/threejs_film && npm install playwright > /dev/null 2>&1
!cd /content/ricks-clips/templates/threejs_film && npx playwright install chromium > /dev/null 2>&1
!cd /content/ricks-clips/templates/threejs_film && npx playwright install-deps chromium > /dev/null 2>&1

# 4. Bake Film Plan
print(f"
[4/6] 📐 Baking film plan for {EPISODE}...")
%cd /content/ricks-clips
!python tools/film_plan.py check episodes/{EPISODE}/film.json

# Read plan details
plan_path = f'episodes/{EPISODE}/out/plan/camera_baked.json'
if os.path.exists(plan_path):
    with open(plan_path, 'r') as f:
        cam_data = json.load(f)
    total_frames = len(cam_data.get('frames', []))
    fps = cam_data.get('fps', 30)
else:
    total_frames = 1821
    fps = 30
print(f"Episode: {total_frames} frames at {fps} fps ({total_frames/fps:.1f} seconds)")

# 5. Parallel Multi-Worker GPU Render
print(f"
[5/6] 🎥 Launching {WORKERS} parallel GPU workers across {total_frames} frames...")
t0 = time.time()
slice_size = total_frames // WORKERS
ranges = []
for i in range(WORKERS):
    start = i * slice_size
    end = (i + 1) * slice_size - 1 if i < WORKERS - 1 else total_frames - 1
    ranges.append((start, end))

procs = []
for i, (start, end) in enumerate(ranges):
    chunk_out = f'/content/chunk_{i}.mp4'
    log_out = f'/content/worker_{i}.log'
    cmd = f'xvfb-run -a -s "-screen 0 1080x1920x24" node templates/threejs_film/render.mjs episodes/{EPISODE}/film.json --animatic --range {start}-{end} --scale {SCALE} --no-audio --out {chunk_out} > {log_out} 2>&1'
    p = subprocess.Popen(cmd, shell=True, cwd='/content/ricks-clips')
    procs.append((i, p, chunk_out, log_out, start, end))
    print(f"  Worker {i+1} dispatched: Frames {start} -> {end}")

# Live progress loop
finished = set()
while len(finished) < len(procs):
    time.sleep(3)
    counts = []
    for i, p, chunk_out, log_out, start, end in procs:
        if p.poll() is not None and i not in finished:
            finished.add(i)
            print(f"  Worker {i+1} completed! (Exit code {p.returncode})")
        dirs = glob.glob(f"/tmp/film_frames_*")
        c = 0
        for d in dirs:
            c += len(glob.glob(d + "/*.png"))
        counts.append(c)
    cur = counts[-1] if counts else 0
    pct = min(100.0, (cur / total_frames) * 100) if total_frames else 0
    print(f"  Rendering in progress: ~{cur}/{total_frames} frames ({pct:.1f}%) | Elapsed: {time.time()-t0:.0f}s", end="\r")

render_time = time.time() - t0
print(f"\nAll {WORKERS} GPU workers finished in {render_time:.1f}s ({render_time/60:.1f}m)!")

# 6. Concatenate & Synchronize Audio
print(f"
[6/6] 🎵 Concatenating chunks and muxing synchronized audio...")
final_video = f"/content/{EPISODE}_full_episode_1080p.mp4"
with open('/content/chunks.txt', 'w') as f:
    for i in range(WORKERS):
        f.write(f"file '/content/chunk_{i}.mp4'\n")

voice_path = f"/content/ricks-clips/episodes/{EPISODE}/voice/voice.wav"
if not os.path.exists(voice_path):
    voice_path = f"/content/ricks-clips/episodes/{EPISODE}/voice/voice.mp3"

if os.path.exists(voice_path):
    concat_cmd = f"ffmpeg -y -f concat -safe 0 -i /content/chunks.txt -i '{voice_path}' -c:v copy -c:a aac -af apad -shortest '{final_video}'"
else:
    concat_cmd = f"ffmpeg -y -f concat -safe 0 -i /content/chunks.txt -c copy '{final_video}'"

subprocess.run(concat_cmd, shell=True, check=True)

# 7. Verification & Instant Download
if os.path.exists(final_video) and os.path.getsize(final_video) > 1000:
    mb = os.path.getsize(final_video) / (1024*1024)
    total_time = time.time() - t0
    print("="*65)
    print(f"🎉 SUCCESS! Full 1080p video ready: {final_video} ({mb:.2f} MB)")
    print(f"⏱️ Total Render Time: {total_time:.1f}s ({total_time/60:.1f} minutes)")
    print("⬇️ Initiating instant download to your PC...")
    print("="*65)
    try:
        files.download(final_video)
    except Exception as e:
        print("Download note:", e)
else:
    print("\n❌ Error: Video generation failed.")
